# Q4 Appendix: Cross-Network Transfer Asymmetry — All Metrics

This appendix reproduces the article-ready Q4 tables on **cross-network transfer
asymmetry** (F1($i\to j$) $-$ F1($j\to i$)) but expands every table to report
**all four metrics** — Accuracy, Precision, Recall, and Macro-F1 (`f1_score`) —
as raw **means over runs**, instead of macro-F1 alone.

**Invariants (identical to `Q4/Q4_analysis.ipynb`):**
- Drop split `Random with same distribution`.
- Cross-network only (`train_set != test_set`).
- Exclude the SetB$\leftrightarrow$SetC pair (same physical network, different capture period).
- All 11 models; direction = `train_set` $\to$ `test_set`; average over `run_no`.

**Expansion rule:** every macro-F1-valued column becomes 4 columns (Accuracy,
Precision, Recall, Macro-F1); one headline derived quantity (asymmetry /
source$-$target / mean$|$asym$|$) is reported on Macro-F1 only; the categorical
"Better source" is computed on Macro-F1. All statistical machinery (Wilcoxon
$p$, Cohen's $d$, significance) is dropped.


## Setup and data loading

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from itertools import combinations
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH = Path('../data/wandb_export_final_hyperparameters.csv')
TAB_DIR   = Path('tables'); TAB_DIR.mkdir(exist_ok=True)

# ── Q4 invariants ─────────────────────────────────────────────────────────────
NETWORKS   = ['SetA', 'SetB', 'SetC', 'SetD']
EXCLUDED_PAIRS     = {('SetB', 'SetC'), ('SetC', 'SetB')}   # directed
EXCLUDED_UNORDERED = {frozenset(('SetB', 'SetC'))}          # unordered
PAIRS      = [p for p in combinations(NETWORKS, 2)
              if frozenset(p) not in EXCLUDED_UNORDERED]    # 5 unordered pairs
ALL_MODELS = ['gru', 'knn', 'lightgbm', 'logistic-regression', 'lstm',
              'mlp', 'nn', 'random-forest', 'rnn', 'svm', 'xgboost']
TASKS   = ['Binary', 'Multiclass']

# The four metrics carried through, with display labels. Macro-F1 == f1_score.
METRICS      = ['accuracy', 'precision', 'recall', 'f1_score']
METRIC_LABEL = {'accuracy': 'Accuracy', 'precision': 'Precision',
                'recall': 'Recall', 'f1_score': 'Macro-F1'}
PRIMARY      = 'f1_score'   # headline derived quantity uses Macro-F1

PAPER_MODEL_NAMES = {
    'gru': 'GRU', 'lstm': 'LSTM', 'rnn': 'RNN', 'mlp': 'MLP', 'nn': 'NN',
    'knn': 'kNN', 'lightgbm': 'LightGBM', 'logistic-regression': 'LR',
    'random-forest': 'RandomForest', 'svm': 'SVM', 'xgboost': 'XGBoost',
}
CAPTION_NOTE = (r'Values are means over runs of Accuracy/Precision/Recall/Macro-F1. '
                r'The SetB$\leftrightarrow$SetC pair is excluded '
                r'(same network, different capture period).')


def drop_excluded_pairs(df, train_col='train_set', test_col='test_set'):
    keep = ~df.apply(lambda r: (r[train_col], r[test_col]) in EXCLUDED_PAIRS, axis=1)
    return df[keep].copy()


print('Setup ready. Pairs analysed:', [f'{a}-{b}' for a, b in PAIRS])


Setup ready. Pairs analysed: ['SetA-SetB', 'SetA-SetC', 'SetA-SetD', 'SetB-SetD', 'SetC-SetD']


The next cell loads the results CSV, melts **all four metrics** into a long format (carrying the `metric` column), keeps cross-network rows only, drops the excluded pair, then averages over runs and pivots so each configuration has one column per metric.

In [2]:
# ── Load raw data ─────────────────────────────────────────────────────────────
raw = pd.read_csv(DATA_PATH, index_col=0)
raw = raw[raw['split'] != "Random with same distribution"]

metric_cols = [c for c in raw.columns if c.count('/') == 2]
id_vars = ['model', 'task', 'split', 'enable_sequences', 'run_no']

long = raw[id_vars + metric_cols].melt(
    id_vars=id_vars, value_vars=metric_cols,
    var_name='metric_key', value_name='value'
)
long[['train_set', 'test_set', 'metric']] = long['metric_key'].str.split('/', expand=True)
long = long.drop(columns='metric_key')

# Cross-network rows only — ALL four metrics carried through (no PRIMARY filter).
cross = long[
    (long['train_set'] != long['test_set']) &
    (long['metric'].isin(METRICS))
].copy()
cross['direction'] = cross['train_set'] + u'→' + cross['test_set']
cross = drop_excluded_pairs(cross)   # exclude SetB<->SetC

print(f'Cross-network rows (all metrics): {len(cross):,}')
print(f'Directed pairs: {cross["direction"].nunique()}')
print(f'Metrics carried: {sorted(cross["metric"].unique())}')

# ── Average across runs, one column per metric ────────────────────────────────
group_keys = ['model', 'task', 'split', 'enable_sequences',
              'train_set', 'test_set', 'direction']
avg_long = (cross.groupby(group_keys + ['metric'], as_index=False)['value']
                 .mean())
avg = (avg_long.pivot(index=group_keys, columns='metric', values='value')
               .reset_index())
avg.columns.name = None
print('Averaged cross-network configs:', len(avg))
print(avg[group_keys + METRICS].head(4).to_string(index=False))


Cross-network rows (all metrics): 24,080
Directed pairs: 10
Metrics carried: ['accuracy', 'f1_score', 'precision', 'recall']
Averaged cross-network configs: 680
model   task        split  enable_sequences train_set test_set direction  accuracy  precision   recall  f1_score
  gru Binary Random split              True      SetA     SetB SetA→SetB  0.935300   0.528152 0.519929  0.522501
  gru Binary Random split              True      SetA     SetC SetA→SetC  0.938150   0.574176 0.600375  0.582726
  gru Binary Random split              True      SetA     SetD SetA→SetD  0.921661   0.805699 0.687819  0.726382
  gru Binary Random split              True      SetB     SetA SetB→SetA  0.645426   0.455472 0.375357  0.403561


## Table q4_appendix_pair_asymmetry

Per unordered pair: F1(src$\to$tgt) and F1(tgt$\to$src) for **each of the 4
metrics** (8 metric columns), plus the Macro-F1 asymmetry (fwd $-$ rev) and the
Macro-F1 "Better source". Sorted by $|$Macro-F1 asymmetry$|$ descending.
*(Source: `table1_asymmetry_per_pair`.)*


In [3]:
# ── Table 1: Asymmetry per unordered pair (all metrics) ───────────────────────
pair_means = (avg.groupby(['train_set', 'test_set'], as_index=False)[METRICS].mean())

records = []
for net1, net2 in PAIRS:
    fwd = pair_means[(pair_means['train_set'] == net1) & (pair_means['test_set'] == net2)]
    rev = pair_means[(pair_means['train_set'] == net2) & (pair_means['test_set'] == net1)]
    if len(fwd) == 0 or len(rev) == 0:
        continue
    rec = {'Pair': f'{net1} ↔ {net2}'}
    for m in METRICS:
        rec[f'{METRIC_LABEL[m]} (src→tgt)'] = round(float(fwd[m].values[0]), 4)
    for m in METRICS:
        rec[f'{METRIC_LABEL[m]} (tgt→src)'] = round(float(rev[m].values[0]), 4)
    asym = float(fwd[PRIMARY].values[0]) - float(rev[PRIMARY].values[0])
    rec['Asymmetry MacroF1'] = round(asym, 4)
    rec['Better source'] = net1 if asym > 0 else (net2 if asym < 0 else 'Tie')
    rec['_abs'] = abs(asym)
    records.append(rec)

t1 = (pd.DataFrame(records).sort_values('_abs', ascending=False)
        .drop(columns='_abs').reset_index(drop=True))

print('q4_appendix_pair_asymmetry')
print(t1.to_string(index=False))
t1.to_csv(TAB_DIR / 'q4_appendix_pair_asymmetry.csv', index=False)

latex1 = t1.to_latex(index=False, float_format='%.4f', escape=False,
    caption=(r'Cross-network transfer asymmetry per network pair, all metrics. '
             r'src is the first network in the pair label, tgt is the second. '
             r'Asymmetry MacroF1 = F1(src $\to$ tgt) $-$ F1(tgt $\to$ src); '
             r'sorted by absolute Macro-F1 asymmetry (largest first). ' + CAPTION_NOTE),
    label='tab:q4_appendix_pair_asymmetry')
(TAB_DIR / 'q4_appendix_pair_asymmetry.tex').write_text(latex1)
print('Saved q4_appendix_pair_asymmetry.{csv,tex}')
t1


q4_appendix_pair_asymmetry
       Pair  Accuracy (src→tgt)  Precision (src→tgt)  Recall (src→tgt)  Macro-F1 (src→tgt)  Accuracy (tgt→src)  Precision (tgt→src)  Recall (tgt→src)  Macro-F1 (tgt→src)  Asymmetry MacroF1 Better source
SetA ↔ SetD              0.8569               0.7739            0.6387              0.6464              0.5792               0.4558            0.5531              0.4030             0.2434          SetA
SetC ↔ SetD              0.8207               0.7625            0.6296              0.6150              0.6286               0.4857            0.5574              0.4284             0.1867          SetC
SetB ↔ SetD              0.7909               0.7418            0.6057              0.5747              0.6310               0.4750            0.5619              0.4138             0.1610          SetB
SetA ↔ SetB              0.8485               0.5761            0.4781              0.4825              0.7471               0.4802            0.4679            

,Pair,Accuracy (src→tgt),Precision (src→tgt),Recall (src→tgt),Macro-F1 (src→tgt),Accuracy (tgt→src),Precision (tgt→src),Recall (tgt→src),Macro-F1 (tgt→src),Asymmetry MacroF1,Better source
0,SetA ↔ SetD,0.8569,0.7739,0.6387,0.6464,0.5792,0.4558,0.5531,0.4030,0.2434,SetA
1,SetC ↔ SetD,0.8207,0.7625,0.6296,0.6150,0.6286,0.4857,0.5574,0.4284,0.1867,SetC
2,SetB ↔ SetD,0.7909,0.7418,0.6057,0.5747,0.6310,0.4750,0.5619,0.4138,0.1610,SetB
3,SetA ↔ SetB,0.8485,0.5761,0.4781,0.4825,0.7471,0.4802,0.4679,0.4384,0.0441,SetA
4,SetA ↔ SetC,0.8151,0.5536,0.4901,0.4662,0.7569,0.5197,0.4985,0.4672,-0.0010,SetC


## Table q4_appendix_source_target

Per network: "as Source" and "as Target" for **each of the 4 metrics** (8
columns), plus Source$-$Target on Macro-F1. Sorted by Macro-F1 source rank.
*(Source: `table2_source_target_ranking`.)*


In [4]:
# ── Table 2: Source and target network ranking (all metrics) ──────────────────
src_means = (avg.groupby('train_set', as_index=False)[METRICS].mean()
               .rename(columns={'train_set': 'Network'}))
tgt_means = (avg.groupby('test_set', as_index=False)[METRICS].mean()
               .rename(columns={'test_set': 'Network'}))

rec = {'Network': src_means['Network']}
t2 = pd.DataFrame(rec)
for m in METRICS:
    t2[f'{METRIC_LABEL[m]} (as Source)'] = src_means[m].round(4)
t2 = t2.merge(
    tgt_means.rename(columns={m: f'{METRIC_LABEL[m]} (as Target)' for m in METRICS}),
    on='Network')
for m in METRICS:
    t2[f'{METRIC_LABEL[m]} (as Target)'] = t2[f'{METRIC_LABEL[m]} (as Target)'].round(4)

src_pri, tgt_pri = f'{METRIC_LABEL[PRIMARY]} (as Source)', f'{METRIC_LABEL[PRIMARY]} (as Target)'
t2['Source Rank'] = t2[src_pri].rank(ascending=False).astype(int)
t2['Source - Target MacroF1'] = (t2[src_pri] - t2[tgt_pri]).round(4)
t2 = t2.sort_values('Source Rank').drop(columns='Source Rank').reset_index(drop=True)

print('q4_appendix_source_target')
print(t2.to_string(index=False))
t2.to_csv(TAB_DIR / 'q4_appendix_source_target.csv', index=False)

latex2 = t2.to_latex(index=False, float_format='%.4f', escape=False,
    caption=(r'Per-network cross-network role as training source and evaluation '
             r'target, all metrics (averaged over all models, tasks, splits, '
             r'input representations). Source $-$ Target MacroF1 $>$ 0: network '
             r'generalises outward better than it receives generalisation from '
             r'others; sorted by Macro-F1 source rank. ' + CAPTION_NOTE),
    label='tab:q4_appendix_source_target')
(TAB_DIR / 'q4_appendix_source_target.tex').write_text(latex2)
print('Saved q4_appendix_source_target.{csv,tex}')
t2


q4_appendix_source_target
Network  Accuracy (as Source)  Precision (as Source)  Recall (as Source)  Macro-F1 (as Source)  Accuracy (as Target)  Precision (as Target)  Recall (as Target)  Macro-F1 (as Target)  Source - Target MacroF1
   SetC                0.7888                 0.6411              0.5641                0.5411                0.7219                 0.5197              0.5238                0.4473                   0.0938
   SetA                0.8402                 0.6345              0.5356                0.5317                0.6944                 0.4852              0.5065                0.4362                   0.0955
   SetB                0.7690                 0.6110              0.5368                0.5065                0.7397                 0.5255              0.5200                0.4481                   0.0584
   SetD                0.6129                 0.4722              0.5575                0.4150                0.8228                 0.7594       

,Network,Accuracy (as Source),Precision (as Source),Recall (as Source),Macro-F1 (as Source),Accuracy (as Target),Precision (as Target),Recall (as Target),Macro-F1 (as Target),Source - Target MacroF1
0,SetC,0.7888,0.6411,0.5641,0.5411,0.7219,0.5197,0.5238,0.4473,0.0938
1,SetA,0.8402,0.6345,0.5356,0.5317,0.6944,0.4852,0.5065,0.4362,0.0955
2,SetB,0.7690,0.6110,0.5368,0.5065,0.7397,0.5255,0.5200,0.4481,0.0584
3,SetD,0.6129,0.4722,0.5575,0.4150,0.8228,0.7594,0.6247,0.6120,-0.1970


## Table q4_appendix_per_model

Per model: Mean Cross-net for **each of the 4 metrics** (4 columns), plus
Macro-F1 Best/Worst source, and Macro-F1 mean$|$asym$|$ / max$|$asym$|$. Sorted
by Macro-F1 mean$|$asym$|$ descending. *(Source: `table3_per_model_asymmetry`.)*


In [5]:
# ── Table 3: Per-model asymmetry (all metrics) ────────────────────────────────
model_records = []
for model in ALL_MODELS:
    sub = avg[avg['model'] == model]
    src_rank = sub.groupby('train_set')[PRIMARY].mean().sort_values(ascending=False)
    asym_vals = []
    for net1, net2 in PAIRS:
        f12 = sub.loc[(sub['train_set'] == net1) & (sub['test_set'] == net2), PRIMARY].mean()
        f21 = sub.loc[(sub['train_set'] == net2) & (sub['test_set'] == net1), PRIMARY].mean()
        if not (np.isnan(f12) or np.isnan(f21)):
            asym_vals.append(abs(f12 - f21))
    rec = {'model': PAPER_MODEL_NAMES.get(model, model)}
    for m in METRICS:
        rec[f'Mean Cross-net {METRIC_LABEL[m]}'] = round(sub[m].mean(), 4)
    rec['Best source'] = src_rank.index[0] if len(src_rank) else '-'
    rec['Worst source'] = src_rank.index[-1] if len(src_rank) else '-'
    rec['Mean |asym| MacroF1'] = round(np.mean(asym_vals), 4) if asym_vals else np.nan
    rec['Max |asym| MacroF1'] = round(np.max(asym_vals), 4) if asym_vals else np.nan
    model_records.append(rec)

t3 = (pd.DataFrame(model_records)
        .sort_values('Mean |asym| MacroF1', ascending=False)
        .reset_index(drop=True))

print('q4_appendix_per_model')
print(t3.to_string(index=False))
t3.to_csv(TAB_DIR / 'q4_appendix_per_model.csv', index=False)

latex3 = t3.to_latex(index=False, float_format='%.4f', escape=False,
    caption=(r'Per-model cross-network transfer asymmetry, all metrics. '
             r'Mean Cross-net columns are means over all directed cross-network '
             r'transfers, tasks, splits, input representations, and runs. Best/'
             r'Worst source and mean/max $|$asym$|$ are computed on Macro-F1 '
             r'over the five analysed pairs; sorted by Macro-F1 mean $|$asymmetry'
             r'$|$ descending. ' + CAPTION_NOTE),
    label='tab:q4_appendix_per_model')
(TAB_DIR / 'q4_appendix_per_model.tex').write_text(latex3)
print('Saved q4_appendix_per_model.{csv,tex}')
t3


q4_appendix_per_model
       model  Mean Cross-net Accuracy  Mean Cross-net Precision  Mean Cross-net Recall  Mean Cross-net Macro-F1 Best source Worst source  Mean |asym| MacroF1  Max |asym| MacroF1
     XGBoost                   0.7740                    0.6040                 0.5766                   0.5338        SetC         SetD               0.2100              0.3213
    LightGBM                   0.7765                    0.6055                 0.5905                   0.5450        SetC         SetD               0.1975              0.3302
        LSTM                   0.7459                    0.5573                 0.5389                   0.4600        SetA         SetD               0.1870              0.3242
RandomForest                   0.7681                    0.6054                 0.5924                   0.5422        SetC         SetD               0.1803              0.2952
         MLP                   0.8183                    0.6158                 0.5293  

,model,Mean Cross-net Accuracy,Mean Cross-net Precision,Mean Cross-net Recall,Mean Cross-net Macro-F1,Best source,Worst source,Mean |asym| MacroF1,Max |asym| MacroF1
0,XGBoost,0.7740,0.6040,0.5766,0.5338,SetC,SetD,0.2100,0.3213
1,LightGBM,0.7765,0.6055,0.5905,0.5450,SetC,SetD,0.1975,0.3302
2,LSTM,0.7459,0.5573,0.5389,0.4600,SetA,SetD,0.1870,0.3242
3,RandomForest,0.7681,0.6054,0.5924,0.5422,SetC,SetD,0.1803,0.2952
4,MLP,0.8183,0.6158,0.5293,0.4923,SetC,SetD,0.1406,0.2539
5,RNN,0.8256,0.6033,0.5724,0.5155,SetA,SetD,0.1332,0.2314
6,GRU,0.8358,0.6015,0.5803,0.5258,SetA,SetD,0.1300,0.2648
7,NN,0.7085,0.5667,0.5084,0.4300,SetA,SetC,0.1083,0.2529
8,kNN,0.6409,0.5184,0.5119,0.4358,SetC,SetD,0.1000,0.2058
9,LR,0.7903,0.6093,0.5439,0.5296,SetA,SetD,0.0944,0.1742


## Table q4_appendix_pair_task_split

Raw-means replacement for the statistical table. Index (pair, task, split);
forward and reverse **directional means for each of the 4 metrics** (8 columns),
plus Macro-F1 asymmetry. Directions are matched per (task) on identical configs
(model, split, `enable_sequences`) — the same paired-directions logic as the
source — but plain directional means per metric are reported.
*(Source: `q4_tab_pair_asymmetry` / `table4_statistical_tests` granularity.)*


In [6]:
# ── Table 4: per-(pair, task, split) directional means (all metrics) ──────────
# Match forward/reverse directions on identical configurations (model,
# split, enable_sequences); report plain directional means per metric.
cfg_keys = ['model', 'split', 'enable_sequences']

def paired_directions(sub, net1, net2, keys):
    fwd = sub[(sub['train_set'] == net1) & (sub['test_set'] == net2)]
    rev = sub[(sub['train_set'] == net2) & (sub['test_set'] == net1)]
    fwd = fwd[keys + METRICS].rename(columns={m: f'{m}_fwd' for m in METRICS})
    rev = rev[keys + METRICS].rename(columns={m: f'{m}_rev' for m in METRICS})
    return fwd.merge(rev, on=keys, how='inner')

rows = []
for net1, net2 in PAIRS:
    for task in TASKS:
        sub = avg[avg['task'] == task]
        m = paired_directions(sub, net1, net2, cfg_keys)
        if len(m) < 2:
            continue
        # Report per split (split is a matched key, retained by its own name).
        for split, g in m.groupby('split'):
            rec = {'pair': f'{net1}↔{net2}', 'task': task, 'split': split}
            for me in METRICS:
                rec[f'{METRIC_LABEL[me]} (fwd)'] = round(g[f'{me}_fwd'].mean(), 4)
            for me in METRICS:
                rec[f'{METRIC_LABEL[me]} (rev)'] = round(g[f'{me}_rev'].mean(), 4)
            rec['Asymmetry MacroF1'] = round(
                g[f'{PRIMARY}_fwd'].mean() - g[f'{PRIMARY}_rev'].mean(), 4)
            rec['_abs'] = abs(rec['Asymmetry MacroF1'])
            rows.append(rec)

pts = pd.DataFrame(rows)
# Sort: pairs by mean |MacroF1 asym| desc, then task, then split (source order).
pair_order = (pts.groupby('pair')['_abs'].mean()
              .sort_values(ascending=False).index.tolist())
pts['pair'] = pd.Categorical(pts['pair'], categories=pair_order, ordered=True)
pts = (pts.sort_values(['pair', 'task', 'split'])
          .drop(columns='_abs').reset_index(drop=True))

print('q4_appendix_pair_task_split')
print(pts.to_string(index=False))
pts.to_csv(TAB_DIR / 'q4_appendix_pair_task_split.csv', index=False)

latex4 = pts.to_latex(index=False, float_format='%.4f', escape=False,
    caption=(r'Per-(pair, task, split) directional means of cross-network '
             r'transfer, all metrics. For each unordered pair and task, forward '
             r'and reverse directions are matched on identical configurations '
             r'(model, split, input representation); reported values are plain '
             r'directional means per metric. Asymmetry MacroF1 = forward $-$ '
             r'reverse mean Macro-F1; pairs sorted by mean absolute Macro-F1 '
             r'asymmetry. ' + CAPTION_NOTE),
    label='tab:q4_appendix_pair_task_split')
(TAB_DIR / 'q4_appendix_pair_task_split.tex').write_text(latex4)
print('Saved q4_appendix_pair_task_split.{csv,tex}')
pts


q4_appendix_pair_task_split
     pair       task        split  Accuracy (fwd)  Precision (fwd)  Recall (fwd)  Macro-F1 (fwd)  Accuracy (rev)  Precision (rev)  Recall (rev)  Macro-F1 (rev)  Asymmetry MacroF1
SetA↔SetD     Binary Random split          0.8707           0.8231        0.7093          0.7280          0.5375           0.5160        0.5056          0.4296             0.2984
SetA↔SetD     Binary   Time split          0.8593           0.8625        0.6964          0.6879          0.6448           0.5493        0.6600          0.4853             0.2026
SetA↔SetD Multiclass Random split          0.8382           0.7301        0.5641          0.5903          0.4888           0.3656        0.4609          0.3162             0.2741
SetA↔SetD Multiclass   Time split          0.8595           0.6800        0.5850          0.5793          0.6456           0.3922        0.5861          0.3808             0.1986
SetC↔SetD     Binary Random split          0.8104           0.8024        0.6

,pair,task,split,Accuracy (fwd),Precision (fwd),Recall (fwd),Macro-F1 (fwd),Accuracy (rev),Precision (rev),Recall (rev),Macro-F1 (rev),Asymmetry MacroF1
0,SetA↔SetD,Binary,Random split,0.8707,0.8231,0.7093,0.7280,0.5375,0.5160,0.5056,0.4296,0.2984
1,SetA↔SetD,Binary,Time split,0.8593,0.8625,0.6964,0.6879,0.6448,0.5493,0.6600,0.4853,0.2026
2,SetA↔SetD,Multiclass,Random split,0.8382,0.7301,0.5641,0.5903,0.4888,0.3656,0.4609,0.3162,0.2741
3,SetA↔SetD,Multiclass,Time split,0.8595,0.6800,0.5850,0.5793,0.6456,0.3922,0.5861,0.3808,0.1986
4,SetC↔SetD,Binary,Random split,0.8104,0.8024,0.6970,0.6811,0.6721,0.5635,0.6936,0.5152,0.1659
5,SetC↔SetD,Binary,Time split,0.8695,0.8913,0.7367,0.7434,0.6611,0.5699,0.6551,0.5055,0.2379
6,SetC↔SetD,Multiclass,Random split,0.7738,0.6026,0.5119,0.4830,0.5589,0.3677,0.4321,0.3166,0.1664
7,SetC↔SetD,Multiclass,Time split,0.8291,0.7538,0.5730,0.5526,0.6224,0.4419,0.4489,0.3761,0.1765
8,SetB↔SetD,Binary,Random split,0.8490,0.8740,0.6870,0.6958,0.6732,0.5787,0.6848,0.5075,0.1883
9,SetB↔SetD,Binary,Time split,0.7723,0.8326,0.7123,0.6739,0.6566,0.5343,0.6596,0.4732,0.2007
